# Guardrailed Multi-Agent System: run and evidence notebook

This notebook runs the full system on a Google Colab and records the evidence for the submission.

* **Part 1** prepares the Colab computer (download the project, install libraries, add the key, run the tests).
* **Part 2** is session 1: save organisation facts, run a normal job, show the attempt cap and the kill-switch.
* **Part 3** is session 2, after a restart: prove the memory survived and is used.
* **Part 4** shows the same engine reused for a different kind of job.

Run the cells from top to bottom. Every command runs as a separate program, so each one is also a fresh session.

## Part 1: prepare the Colab computer

### 1.1 Download (clone) the project from GitHub
Change `GITHUB_USER` if your GitHub username is different.

In [1]:
GITHUB_USER = "mapanya"
REPO_NAME = "guardrailed-multi-agent-system"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

import os
os.chdir("/content")
if os.path.isdir(REPO_NAME):
    !cd {REPO_NAME} && git pull
else:
    !git clone {REPO_URL}
os.chdir(f"/content/{REPO_NAME}")
!pwd && ls

Cloning into 'guardrailed-multi-agent-system'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 33 (delta 0), reused 33 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 48.26 KiB | 6.89 MiB/s, done.
/content/guardrailed-multi-agent-system
config		     pyproject.toml	   requirements.txt
design_decisions.md  readme.md		   src
notebooks	     requirements-dev.txt  tests


### 1.2 Install the libraries
This takes 2 to 4 minutes. Warnings about other Colab packages can be ignored.

In [2]:
!python --version
!pip install -q -r requirements.txt
!pip install -q -e .
print("Install finished")

Python 3.13.15
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.1/308.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 70.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━

### 1.3 Add your Groq key
The cell first looks for a Colab secret called `GROQ_API_KEY`. If it cannot find one, it asks you to paste the key into a hidden box. The key is kept only in this session and is never written to a file.

In [3]:
import os
from getpass import getpass

def load_key():
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return getpass("Paste your Groq key and press Enter: ")

os.environ["GROQ_API_KEY"] = load_key().strip()
print("Key loaded:", "yes" if os.environ["GROQ_API_KEY"] else "no")

Key loaded: yes


### 1.4 Run the automated tests
These check the guardrails, the output forms and the workflow without using the key. All should pass.

In [4]:
!python -m pytest -q

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0
rootdir: /content/guardrailed-multi-agent-system
configfile: pyproject.toml
testpaths: tests
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 33 items                                                             

tests/test_guardrailed_mas.py .................................          [100%]

============================== 33 passed in 1.41s ==============================


## Part 2: session 1

### 2.1 Save organisation facts to memory
These facts are stored in the SQLite history file and the local search index inside the `data` folder.

In [5]:
USER_ID = "demo_user"
!python -m guardrailed_mas.main remember --user-id {USER_ID} --fact "Our email platform is Microsoft 365 with Defender for Office 365."
!python -m guardrailed_mas.main remember --user-id {USER_ID} --fact "All security incidents are escalated to the SOC Lead and the IT Manager."

/usr/local/lib/python3.13/dist-packages/jupyter_client/jsonutil.py:32: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  datetime.strptime("1", "%d")
21:14:41 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
21:14:47 INFO Memory ready. SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db
Saved for demo_user: Our email platform is Microsoft 365 with Defender for Office 365.
SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db (1 row(s))
/usr/local/lib/python3.13/dist-packages/jupyter_client/jsonutil.py:32: DeprecationWarni

### 2.2 Normal run: phishing incident
Watch the step lines: `planner`, `budget_gate` with `execution_count`, `executor`, `reviewer` and finally `finalize` (approved) or `fallback` (halted).

In [6]:
!python -m guardrailed_mas.main run --scenario phishing --user-id {USER_ID}

21:16:53 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
21:16:56 INFO Memory ready. SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db
21:16:56 INFO Starting job for user demo_user with profile security_incident (cap=3)
21:16:58 INFO node=recall_memory user_id='demo_user' facts_found=2
21:17:01 INFO node=planner valid=True validated_by='direct' steps=5
21:17:01 INFO node=budget_gate execution_count=1 max_executions=3 exceeded=False
21:17:03 INFO node=executor attempt=1 valid=True validated_by='direct' items=7
21:17:03 INFO node=reviewer approved=True score=9 validated_by='direct' execution_count=1
21:17:03 INFO node=finalize status='approved' attempts_used=1
21:17:03 INFO node=save_memory saved=True

================ FINAL ANSWER ================
{
  "status": "approved",
  "task": "A finance officer reports a suspicious email that asks them to update suppli

### 2.3 Guardrail demonstration: the Reviewer rejects everything
The Executor is allowed 3 attempts. When `execution_count` reaches 4 (more than 3), the graph stops at once and returns the fixed fallback answer.

In [7]:
!python -m guardrailed_mas.main run --scenario ransomware --user-id {USER_ID} --demo-guardrail

21:22:32 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
21:22:35 INFO Memory ready. SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db
21:22:35 INFO Starting job for user demo_user with profile security_incident (cap=3)
21:22:37 INFO node=recall_memory user_id='demo_user' facts_found=3
21:22:40 INFO node=planner valid=True validated_by='direct' steps=6
21:22:40 INFO node=budget_gate execution_count=1 max_executions=3 exceeded=False
21:22:42 INFO node=executor attempt=1 valid=True validated_by='direct' items=7
21:22:42 INFO node=reviewer approved=False score=0 validated_by='direct' execution_count=1
21:22:42 INFO node=budget_gate execution_count=2 max_executions=3 exceeded=False
21:22:43 INFO node=executor attempt=2 valid=True validated_by='direct' items=7
21:22:43 INFO node=reviewer approved=False score=0 validated_by='direct' execution_count=2
21:22:43 INFO

### 2.4 Kill-switch demonstration
With the emergency stop switched on, no worker is called at all.

In [8]:
!MAS_KILL_SWITCH=1 python -m guardrailed_mas.main run --scenario lost_laptop --user-id {USER_ID}

21:25:32 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
21:25:35 INFO Memory ready. SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db
21:25:35 INFO Starting job for user demo_user with profile security_incident (cap=3)
21:25:37 INFO node=recall_memory user_id='demo_user' facts_found=4
21:25:37 INFO node=planner halted='kill_switch'
21:25:37 INFO node=fallback status='halted' reason='kill_switch' attempts_used=0 execution_count=0
21:25:37 INFO node=save_memory saved=True

================ FINAL ANSWER ================
{
  "status": "halted",
  "reason": "kill_switch",
  "message": "The automated workflow was stopped by a safety control before it could produce an approved answer. No further automated actions were taken. Please hand this job to a human analyst using your standard escalation procedure.",
  "task": "A sales manager reports that their company lap

## Part 3: session 2 (after a restart)

**Now restart the session:** in Colab choose *Runtime > Restart session*. In VS Code click **Restart** on the notebook toolbar.

A restart wipes everything held in the notebook's working memory, including the key and all variables. The `data` folder on the Colab computer is kept. That is where Mem0 stored its SQLite file. Then run the cells below.

In [1]:
import os
from getpass import getpass
os.chdir("/content/guardrailed-multi-agent-system")

def load_key():
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return getpass("Paste your Groq key and press Enter: ")

os.environ["GROQ_API_KEY"] = load_key().strip()
USER_ID = "demo_user"
print("Session 2 ready")

Session 2 ready


### 3.1 What does memory hold now?
The facts from session 1 and short run summaries are still there.

In [2]:
!python -m guardrailed_mas.main recall --user-id {USER_ID}

/usr/local/lib/python3.13/dist-packages/jupyter_client/jsonutil.py:32: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  datetime.strptime("1", "%d")
21:33:24 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
21:33:27 INFO Memory ready. SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db
Memory for demo_user (5 item(s)):
  - Our email platform is Microsoft 365 with Defender for Office 365.
  - All security incidents are escalated to the SOC Lead and the IT Manager.
  - Completed job 'A finance officer reports a suspicious email that asks them to upda

### 3.2 Look inside the SQLite file directly

In [3]:
import sqlite3
db = "data/mem0_history.db"
with sqlite3.connect(db) as con:
    rows = con.execute("SELECT event, substr(new_memory, 1, 90), created_at FROM history ORDER BY created_at").fetchall()
print(f"{len(rows)} row(s) in {db}")
for row in rows:
    print(row)

5 row(s) in data/mem0_history.db
('ADD', 'Our email platform is Microsoft 365 with Defender for Office 365.', '2026-09-26T21:14:47.341006+00:00')
('ADD', 'All security incidents are escalated to the SOC Lead and the IT Manager.', '2026-09-26T21:15:12.553738+00:00')
('ADD', "Completed job 'A finance officer reports a suspicious email that asks them to update suppl", '2026-09-26T21:17:03.808642+00:00')
('ADD', "Job 'Several file shares on the head office server show renamed files and a ransom note. S", '2026-09-26T21:22:45.605099+00:00')
('ADD', "Job 'A sales manager reports that their company laptop was stolen from a car. The laptop h", '2026-09-26T21:25:37.246826+00:00')


### 3.3 New job that uses the remembered facts
In the step lines, `recall_memory` shows `facts_found` above 0 and the checklist should mention Microsoft 365 and the SOC lead without being told again.

In [4]:
!python -m guardrailed_mas.main run --scenario account_takeover --user-id {USER_ID}

21:38:46 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
21:38:49 INFO Memory ready. SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db
21:38:49 INFO Starting job for user demo_user with profile security_incident (cap=3)
21:38:51 INFO node=recall_memory user_id='demo_user' facts_found=5
21:38:53 INFO node=planner valid=True validated_by='direct' steps=6
21:38:53 INFO node=budget_gate execution_count=1 max_executions=3 exceeded=False
21:38:55 INFO node=executor attempt=1 valid=True validated_by='direct' items=7
21:38:57 INFO node=reviewer approved=False score=4 validated_by='direct' execution_count=1
21:38:57 INFO node=budget_gate execution_count=2 max_executions=3 exceeded=False
21:38:58 INFO node=executor attempt=2 valid=True validated_by='direct' items=6
21:39:02 INFO node=reviewer approved=True score=9 validated_by='direct' execution_count=2
21:39:02 INFO 

## Part 4: reuse for a different job
The engine is unchanged. Only the worker profile in `config/agents.yaml` differs.

In [5]:
!python -m guardrailed_mas.main run --scenario new_customer --user-id {USER_ID}

21:42:27 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
21:42:30 INFO Memory ready. SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db
21:42:30 INFO Starting job for user demo_user with profile customer_onboarding (cap=3)
21:42:32 INFO node=recall_memory user_id='demo_user' facts_found=5
21:42:35 INFO node=planner valid=True validated_by='direct' steps=8
21:42:35 INFO node=budget_gate execution_count=1 max_executions=3 exceeded=False
21:42:37 INFO node=executor attempt=1 valid=True validated_by='direct' items=8
21:42:38 INFO node=reviewer approved=True score=9 validated_by='direct' execution_count=1
21:42:38 INFO node=finalize status='approved' attempts_used=1
21:42:38 INFO node=save_memory saved=True

================ FINAL ANSWER ================
{
  "status": "approved",
  "task": "A new corporate customer has signed a managed security services contract a

You can also describe any job in your own words:

In [6]:
!python -m guardrailed_mas.main run --task "A contractor reports that a USB drive with project files is missing." --user-id {USER_ID}

21:43:06 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
21:43:09 INFO Memory ready. SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db
21:43:09 INFO Starting job for user demo_user with profile security_incident (cap=3)
21:43:11 INFO node=recall_memory user_id='demo_user' facts_found=5
21:43:14 INFO node=planner valid=True validated_by='direct' steps=7
21:43:14 INFO node=budget_gate execution_count=1 max_executions=3 exceeded=False
21:43:16 INFO node=executor attempt=1 valid=True validated_by='direct' items=7
21:43:17 INFO node=reviewer approved=True score=9 validated_by='direct' execution_count=1
21:43:17 INFO node=finalize status='approved' attempts_used=1
21:43:17 INFO node=save_memory saved=True

================ FINAL ANSWER ================
{
  "status": "approved",
  "task": "A contractor reports that a USB drive with project files is missing.",
  "at

## Part 5: collect the evidence
This prints every step log. Save this notebook with its outputs (File > Save).

In [7]:
import glob
for path in sorted(glob.glob("logs/run_*.jsonl")):
    print("=" * 20, path)
    print(open(path).read())

==================== logs/run_20260926_211633.jsonl
{"time": "2026-09-26T21:16:56+00:00", "level": "INFO", "logger": "guardrailed_mas.memory", "message": "Memory ready. SQLite history file: /content/guardrailed-multi-agent-system/data/mem0_history.db"}
{"time": "2026-09-26T21:16:56+00:00", "level": "INFO", "logger": "guardrailed_mas.main", "message": "Starting job for user demo_user with profile security_incident (cap=3)"}
{"time": "2026-09-26T21:16:58+00:00", "level": "INFO", "logger": "guardrailed_mas.graph", "node": "recall_memory", "user_id": "demo_user", "facts_found": 2}
{"time": "2026-09-26T21:17:01+00:00", "level": "INFO", "logger": "guardrailed_mas.graph", "node": "planner", "valid": true, "validated_by": "direct", "steps": 5}
{"time": "2026-09-26T21:17:01+00:00", "level": "INFO", "logger": "guardrailed_mas.graph", "node": "budget_gate", "execution_count": 1, "max_executions": 3, "exceeded": false}
{"time": "2026-09-26T21:17:03+00:00", "level": "INFO", "logger": "guardrailed_m